# HumanLM Model Inspection

Notebook ini dipakai untuk memahami arsitektur dasar `snap-stanford/humanlm-opinion`.

Fokusnya:
- baca config model
- cek tokenizer dan chat template
- lihat struktur layer utama
- jalankan forward pass kecil
- ambil hidden states dasar


In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "snap-stanford/humanlm-opinion"
DEVICE = "cpu"
TORCH_DTYPE = torch.bfloat16

print("MODEL_NAME:", MODEL_NAME)
print("DEVICE:", DEVICE)
print("TORCH_DTYPE:", TORCH_DTYPE)

## 1. Inspect Config

In [ ]:
cfg = AutoConfig.from_pretrained(MODEL_NAME)

summary = {
    "model_type": getattr(cfg, "model_type", None),
    "architectures": getattr(cfg, "architectures", None),
    "hidden_size": getattr(cfg, "hidden_size", None),
    "intermediate_size": getattr(cfg, "intermediate_size", None),
    "num_hidden_layers": getattr(cfg, "num_hidden_layers", None),
    "num_attention_heads": getattr(cfg, "num_attention_heads", None),
    "num_key_value_heads": getattr(cfg, "num_key_value_heads", None),
    "vocab_size": getattr(cfg, "vocab_size", None),
    "max_position_embeddings": getattr(cfg, "max_position_embeddings", None),
    "rope_theta": getattr(cfg, "rope_theta", None),
    "torch_dtype": str(getattr(cfg, "torch_dtype", None)),
}

summary

In [ ]:
print(json.dumps(cfg.to_dict(), indent=2, default=str)[:4000])

## 2. Inspect Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_info = {
    "tokenizer_class": tokenizer.__class__.__name__,
    "vocab_size": tokenizer.vocab_size,
    "pad_token": tokenizer.pad_token,
    "eos_token": tokenizer.eos_token,
    "bos_token": tokenizer.bos_token,
    "chat_template_exists": bool(getattr(tokenizer, "chat_template", None)),
    "special_tokens_map": tokenizer.special_tokens_map,
}

tokenizer_info

In [ ]:
if getattr(tokenizer, "chat_template", None):
    print(tokenizer.chat_template[:2000])
else:
    print("No chat template found.")

## 3. Load Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=TORCH_DTYPE,
    device_map=DEVICE,
)
model.eval()

print(type(model).__name__)
print("parameter_count:", sum(p.numel() for p in model.parameters()))

## 4. Inspect Top-Level Structure

In [ ]:
for name, module in model.named_children():
    print(name, '->', type(module).__name__)

In [ ]:
layer_names = []
for name, module in model.named_modules():
    if "layers" in name and name.count(".") <= 3:
        layer_names.append((name, type(module).__name__))

layer_names[:80]

## 5. Forward Pass Kecil

In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}
print({k: tuple(v.shape) for k, v in inputs.items()})

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True, return_dict=True)

print("logits shape:", tuple(outputs.logits.shape))
print("num hidden_states:", len(outputs.hidden_states))
print("embedding output shape:", tuple(outputs.hidden_states[0].shape))
print("last hidden state shape:", tuple(outputs.hidden_states[-1].shape))

## 6. Cek Hidden State per Layer

In [ ]:
hidden_state_shapes = [tuple(h.shape) for h in outputs.hidden_states]
hidden_state_shapes[:10], hidden_state_shapes[-3:]

In [ ]:
last_token_vectors = [h[0, -1, :].float() for h in outputs.hidden_states]
print("num layer vectors:", len(last_token_vectors))
print("vector dim:", last_token_vectors[-1].shape[0])

## 7. Generate Pendek

In [ ]:
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=40, do_sample=False)

new_tokens = generated[0][inputs['input_ids'].shape[-1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))